In [ ]:
# ============================================================
# BreastDM 2D patient-level splitter: 70% / 15% / 15%
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import csv
import filecmp
import math
import random
import shutil
import tempfile
import zipfile

from collections import defaultdict
from pathlib import Path, PurePosixPath
from tqdm.auto import tqdm


# ============================================================
# EDIT THESE TWO PATHS
# ============================================================

# Original BreastDM ZIP in Google Drive
SOURCE_ZIP = Path(
    "/content/drive/MyDrive/BreastDM_Project/data/BreastDMDS.zip"
)

# Where the completed split ZIP should be saved
OUTPUT_ZIP = Path(
    "/content/drive/MyDrive/BreastDM_Project/data/segmentation_DS_7_14_2026.zip"
)

SEED = 42

# Set this to True to check the split without copying files.
DRY_RUN = False


# ============================================================
# SETTINGS
# ============================================================

RATIOS = {
    "train": 0.70,
    "val": 0.15,
    "test": 0.15,
}

SPLITS = ("train", "val", "test")


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_patient_class(patient_id):
    """Determine whether a patient is benign or malignant."""

    if "-Be-" in patient_id:
        return "benign"

    if "-Ma-" in patient_id:
        return "malignant"

    raise ValueError(
        f"Could not determine class from patient ID: {patient_id}"
    )


def allocate_counts(total, ratios):
    """
    Convert percentage ratios into whole patient counts while
    ensuring that every patient is assigned exactly once.
    """

    exact_counts = [total * ratio for ratio in ratios]
    counts = [math.floor(value) for value in exact_counts]

    patients_remaining = total - sum(counts)

    remainder_order = sorted(
        range(len(ratios)),
        key=lambda index: exact_counts[index] - counts[index],
        reverse=True,
    )

    for index in remainder_order[:patients_remaining]:
        counts[index] += 1

    return counts


def extract_seg_folder(source_zip, extraction_root):
    """Extract only the prepared 2D seg folder from the ZIP."""

    print("Reading ZIP file...")

    with zipfile.ZipFile(source_zip, "r") as archive:
        seg_members = []

        for member in archive.infolist():
            parts = PurePosixPath(member.filename).parts

            # Select seg, but not seg3D.
            if "seg" in parts and "seg3D" not in parts:
                if ".." in parts:
                    raise ValueError(
                        f"Unsafe path found in ZIP: {member.filename}"
                    )

                seg_members.append(member)

        if not seg_members:
            raise FileNotFoundError(
                "Could not find the 2D seg folder inside the ZIP."
            )

        print(f"Extracting {len(seg_members):,} seg entries...")

        for member in tqdm(seg_members):
            archive.extract(member, extraction_root)

    # Locate the extracted seg folder.
    candidates = []

    for path in extraction_root.rglob("seg"):
        valid = all(
            (path / split / "images").is_dir()
            and (path / split / "labels").is_dir()
            for split in SPLITS
        )

        if valid:
            candidates.append(path)

    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected one valid seg folder, found {len(candidates)}."
        )

    return candidates[0]


def collect_unique_pairs(seg_root):
    """
    Find every JPG image and its PNG mask.

    Duplicate samples from the released splits are retained only
    when the image and mask files are byte-identical.
    """

    pairs = {}

    for original_split in SPLITS:
        images_root = seg_root / original_split / "images"
        labels_root = seg_root / original_split / "labels"

        image_paths = sorted(images_root.rglob("*.jpg"))

        print(
            f"Scanning original {original_split}: "
            f"{len(image_paths):,} images"
        )

        for image_path in tqdm(image_paths):
            relative_path = image_path.relative_to(images_root)

            # Expected:
            # patient / sequence / slice.jpg
            if len(relative_path.parts) != 3:
                raise ValueError(
                    f"Unexpected image structure: {relative_path}"
                )

            patient_id, sequence, image_name = relative_path.parts
            slice_name = Path(image_name).stem

            mask_path = (
                labels_root
                / patient_id
                / sequence
                / f"{slice_name}.png"
            )

            if not mask_path.is_file():
                raise FileNotFoundError(
                    f"Missing mask for image:\n{image_path}"
                )

            unique_key = (
                patient_id,
                sequence,
                slice_name,
            )

            record = {
                "patient_id": patient_id,
                "class_name": get_patient_class(patient_id),
                "sequence": sequence,
                "slice_name": slice_name,
                "image_path": image_path,
                "mask_path": mask_path,
                "original_image": image_path.relative_to(
                    seg_root
                ).as_posix(),
                "original_mask": mask_path.relative_to(
                    seg_root
                ).as_posix(),
            }

            if unique_key not in pairs:
                pairs[unique_key] = record
                continue

            previous = pairs[unique_key]

            same_image = filecmp.cmp(
                previous["image_path"],
                image_path,
                shallow=False,
            )

            same_mask = filecmp.cmp(
                previous["mask_path"],
                mask_path,
                shallow=False,
            )

            if not same_image or not same_mask:
                raise ValueError(
                    "Found conflicting duplicate files for:\n"
                    f"{unique_key}"
                )

    return list(pairs.values())


def create_patient_split(records, seed):
    """Create a class-stratified patient-level split."""

    patients_by_class = defaultdict(set)

    for record in records:
        patients_by_class[record["class_name"]].add(
            record["patient_id"]
        )

    rng = random.Random(seed)

    assignments = {
        "train": [],
        "val": [],
        "test": [],
    }

    ratios = [RATIOS[split] for split in SPLITS]

    for class_name in ("benign", "malignant"):
        patients = sorted(patients_by_class[class_name])
        rng.shuffle(patients)

        counts = allocate_counts(
            len(patients),
            ratios,
        )

        start = 0

        for split, count in zip(SPLITS, counts):
            assignments[split].extend(
                patients[start : start + count]
            )

            start += count

    # Verify no patient appears in multiple splits.
    patient_sets = {
        split: set(assignments[split])
        for split in SPLITS
    }

    assert not (
        patient_sets["train"] & patient_sets["val"]
    )

    assert not (
        patient_sets["train"] & patient_sets["test"]
    )

    assert not (
        patient_sets["val"] & patient_sets["test"]
    )

    return assignments


def print_split_summary(records, assignments):
    """Print patient and image counts for each split."""

    print("\n================ SPLIT SUMMARY ================\n")

    for split in SPLITS:
        patients = set(assignments[split])

        benign_count = sum(
            "-Be-" in patient
            for patient in patients
        )

        malignant_count = sum(
            "-Ma-" in patient
            for patient in patients
        )

        sample_count = sum(
            record["patient_id"] in patients
            for record in records
        )

        print(
            f"{split.upper():5} | "
            f"{len(patients):3} patients | "
            f"{benign_count:2} benign | "
            f"{malignant_count:3} malignant | "
            f"{sample_count:,} image/mask pairs"
        )

    print("\nPatient overlap: 0")
    print(f"Unique samples: {len(records):,}")


def create_split_dataset(
    records,
    assignments,
    output_root,
):
    """Copy the files into train/val/test images/masks folders."""

    patient_to_split = {}

    for split, patients in assignments.items():
        for patient in patients:
            patient_to_split[patient] = split

    for split in SPLITS:
        (output_root / split / "images").mkdir(
            parents=True,
            exist_ok=True,
        )

        (output_root / split / "masks").mkdir(
            parents=True,
            exist_ok=True,
        )

    manifest_path = output_root / "split_manifest.csv"

    with manifest_path.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as manifest_file:

        columns = [
            "split",
            "patient_id",
            "class_name",
            "sequence",
            "image",
            "mask",
            "original_image",
            "original_mask",
        ]

        writer = csv.DictWriter(
            manifest_file,
            fieldnames=columns,
        )

        writer.writeheader()

        print("\nCopying image/mask pairs...")

        for record in tqdm(records):
            split = patient_to_split[
                record["patient_id"]
            ]

            filename_stem = (
                f'{record["patient_id"]}'
                f'__{record["sequence"]}'
                f'__{record["slice_name"]}'
            )

            output_image = (
                output_root
                / split
                / "images"
                / f"{filename_stem}.jpg"
            )

            output_mask = (
                output_root
                / split
                / "masks"
                / f"{filename_stem}.png"
            )

            if output_image.exists() or output_mask.exists():
                raise FileExistsError(
                    f"Output filename collision: {filename_stem}"
                )

            shutil.copy2(
                record["image_path"],
                output_image,
            )

            shutil.copy2(
                record["mask_path"],
                output_mask,
            )

            writer.writerow({
                "split": split,
                "patient_id": record["patient_id"],
                "class_name": record["class_name"],
                "sequence": record["sequence"],
                "image": output_image.relative_to(
                    output_root
                ).as_posix(),
                "mask": output_mask.relative_to(
                    output_root
                ).as_posix(),
                "original_image": record["original_image"],
                "original_mask": record["original_mask"],
            })


def verify_output(records, output_root):
    """Confirm equal image/mask counts after copying."""

    total_images = 0
    total_masks = 0

    print("\n================ OUTPUT CHECK ================\n")

    for split in SPLITS:
        image_count = len(
            list(
                (output_root / split / "images").glob(
                    "*.jpg"
                )
            )
        )

        mask_count = len(
            list(
                (output_root / split / "masks").glob(
                    "*.png"
                )
            )
        )

        if image_count != mask_count:
            raise AssertionError(
                f"{split} has {image_count} images "
                f"but {mask_count} masks."
            )

        total_images += image_count
        total_masks += mask_count

        print(
            f"{split.upper():5}: "
            f"{image_count:,} images, "
            f"{mask_count:,} masks"
        )

    if total_images != len(records):
        raise AssertionError(
            f"Expected {len(records):,} images, "
            f"found {total_images:,}."
        )

    print("\nVerification passed.")
    print("Every image has a mask.")
    print("No patient appears in multiple splits.")


# ============================================================
# RUN THE COMPLETE WORKFLOW
# ============================================================

if not SOURCE_ZIP.is_file():
    raise FileNotFoundError(
        f"ZIP file does not exist:\n{SOURCE_ZIP}"
    )

if OUTPUT_ZIP.exists():
    raise FileExistsError(
        f"Output ZIP already exists:\n{OUTPUT_ZIP}\n"
        "Choose a new path or remove the previous output."
    )

# Use Colab's local disk for speed.
working_directory = Path(
    tempfile.mkdtemp(
        prefix="breastdm_split_",
        dir="/content",
    )
)

try:
    extraction_root = working_directory / "source"
    local_zip_base = OUTPUT_ZIP.stem # This was changed from output_root.name
    output_root = working_directory / local_zip_base

    extraction_root.mkdir(parents=True)

    print(f'Using temporary working directory: {working_directory}')

    seg_root = extract_seg_folder(SOURCE_ZIP, extraction_root)
    records = collect_unique_pairs(seg_root)
    assignments = create_patient_split(records, SEED)

    print_split_summary(records, assignments)

    if not DRY_RUN:
        create_split_dataset(records, assignments, output_root)
        verify_output(records, output_root)

        output_zip_file = shutil.make_archive(
            output_root,
            "zip",
            output_root.parent,
            output_root.name,
        )

        shutil.move(output_zip_file, OUTPUT_ZIP)
finally:
    # Always clean up the temporary directory.
    print(f"Cleaning up temporary directory: {working_directory}")
    shutil.rmtree(working_directory)

Mounted at /content/drive
Using temporary working directory: /content/breastdm_split_64gcii12
Reading ZIP file...
Extracting 67,494 seg entries...


  0%|          | 0/67494 [00:00<?, ?it/s]

Scanning original train: 20,434 images


  0%|          | 0/20434 [00:00<?, ?it/s]

Scanning original val: 1,989 images


...

Scanning original test: 7,089 images


  0%|          | 0/7089 [00:00<?, ?it/s]


================ SPLIT SUMMARY ================

TRAIN | 162 patients | 59 benign | 103 malignant | 20,417 image/mask pairs
VAL   |  35 patients | 13 benign |  22 malignant | 4,573 image/mask pairs
TEST  |  35 patients | 13 benign |  22 malignant | 4,284 image/mask pairs

Patient overlap: 0
Unique samples: 29,274

Copying image/mask pairs...


  0%|          | 0/29274 [00:00<?, ?it/s]


================ OUTPUT CHECK ================

TRAIN: 20,417 images, 20,417 masks
VAL  : 4,573 images, 4,573 masks
TEST : 4,284 images, 4,284 masks

Verification passed.
Every image has a mask.
No patient appears in multiple splits.
Cleaning up temporary directory: /content/breastdm_split_64gcii12
